In [1]:
import os
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.llms.nvidia import NVIDIA
from llama_index.core import Settings
import chromadb
from dotenv import load_dotenv

c:\Users\Pietro\Documents\develop\rag-mtg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# Load environment variables from .env file
load_dotenv()
NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"]

In [3]:
# Reconnect to the existing persistent Chroma database
# This points to the same path where data was saved in the setup phase
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [4]:
# Get the existing collection (use get_collection, not get_or_create_collection)
# This retrieves the collection that was created and populated earlier
# Will raise an error if the collection doesn't exist
chroma_collection = chroma_client.get_collection("documents_collection")

In [5]:
# Wrap the collection in LlamaIndex's vector store adapter
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [6]:
# Create storage context pointing to the existing vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [12]:
# LLM to generate answers based on retrieved information
Settings.llm = NVIDIA(
    model="meta/llama-3.1-8b-instruct",  # o altri modelli disponibili
    api_key=os.getenv("NVIDIA_API_KEY")
)

# Embedding model to convert text into vectors for retrieval
Settings.embed_model = NVIDIAEmbedding(
    model="nvidia/nv-embed-v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

In [13]:
# Load the index from the existing vector store
# This reads the previously saved embeddings instead of recomputing them
# Note: from_vector_store() instead of from_documents()
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)

In [14]:
# Now you can query the index
# The query engine converts your question to a vector and finds similar document chunks
query_engine = index.as_query_engine()
response = query_engine.query("Can you explain the 'stack'")
print(response)

The stack is a mechanism that keeps track of the order in which spells and/or abilities were added to it. Each time an object is put on the stack, it's put on top of all objects already there. This means that the most recently added object is at the top of the stack, and the oldest object is at the bottom. The stack is used to resolve the effects of spells and abilities in the correct order, ensuring that the game is played fairly and consistently.


In [ ]:
# Optional: print retrieved chunks for debugging
print("\n=== CHUNK RETRIEVED ===")
for i, node in enumerate(response.source_nodes):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Score:  {node.score:.4f}")
    print(f"Source: {node.node.metadata}")
    print(f"Testo:  {node.node.get_content()[:300]}...")  # primi 300 caratteri


=== CHUNK RETRIEVED ===

--- Chunk 1 ---
Score:  0.2715
Source: {'file_path': 'c:\\Users\\Pietro\\Documents\\develop\\rag-mtg\\documents\\MagicCompRules 20260417.txt', 'file_name': 'MagicCompRules 20260417.txt', 'file_type': 'text/plain', 'file_size': 971061, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-01'}
Testo:  That library’s owner doesn’t reveal the order in which the cards go into the library.

401.5. Some effects tell a player to play with the top card of their library revealed, or say that a player may look at the top card of their library. If the top card of the player’s library changes while a spel...

--- Chunk 2 ---
Score:  0.2495
Source: {'file_path': 'c:\\Users\\Pietro\\Documents\\develop\\rag-mtg\\documents\\MagicCompRules 20260417.txt', 'file_name': 'MagicCompRules 20260417.txt', 'file_type': 'text/plain', 'file_size': 971061, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-01'}
Testo:  107.3. Many objects use the letter X as a placehold